# I. Data ingestion
## 1. Creating catalog, schema and volume for bronze layer

* **Objective:** Import raw IEEE-CIS-fraud CSV files into Delta tables managed by Unity Catalog, without any transformation in the bronze layer.

* inputs (the 2 CSV files in the volume)

* outputs (the 2 tables named fraud.bronze.*_raw)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fraud;
CREATE SCHEMA IF NOT EXISTS fraud.bronze;
CREATE VOLUME IF NOT EXISTS fraud.bronze.raw_files
COMMENT 'Landing zone for raw transaction and identity files';

In [0]:
%sql
-- Verification
SHOW CATALOGS;               
SHOW SCHEMAS IN fraud;       
SHOW VOLUMES IN fraud.bronze; 

database,volume_name
bronze,raw_files


In [0]:
%sql
-- Files Verification in the volume
LIST '/Volumes/fraud/bronze/raw_files/';

path,name,size,modification_time
/Volumes/fraud/bronze/raw_files/train_identity.csv,train_identity.csv,26529680,1793045896000
/Volumes/fraud/bronze/raw_files/train_transaction.csv,train_transaction.csv,683351067,1793045823000


## 2. Loading data and transforming into Delta tables

In [0]:
# 1. Loading the "transaction.csv" file and transforming into Delta
df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/fraud/bronze/raw_files/train_transaction.csv")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud.bronze.transactions_raw")
print("transaction.csv ingested successfully")

transaction.csv ingested successfully


In [0]:
# 2. Loading the "identity.csv" file and transforming into Delta
df_identity = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/fraud/bronze/raw_files/train_identity.csv")

df_identity.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud.bronze.identity_raw")

print("train_identity.csv ingested successfully")

train_identity.csv ingested successfully


## 3-Ingested tables verification

In [0]:
%sql
-- Show tables in the bronze layer
SHOW TABLES IN fraud.bronze;



database,tableName,isTemporary
bronze,identity_raw,false
bronze,transactions_raw,false


In [0]:
%sql
-- Counting the number of rows in each table
SELECT 'transactions_raw' AS table_name, COUNT(*) AS nb_rows FROM fraud.bronze.transactions_raw
UNION ALL
SELECT 'identity_raw', COUNT(*) FROM fraud.bronze.identity_raw;



table_name,nb_rows
transactions_raw,590540
identity_raw,144233


In [0]:
# Number of columns in each table
print("Number of columns in the transaction table: ", len(df.columns))
print("Number of columns in the identity table: ", len(df_identity.columns))

Number of columns in the transaction table:  394
Number of columns in the identity table:  41


We have 590,540 rows and 394 columns in the tansaction table and  144,233 rows with 41 columns in the identity table

In [0]:
%sql
-- Overview of transaction table
SELECT * FROM fraud.bronze.transactions_raw LIMIT 5;

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
3356197,0,9172799,26.95,W,15497,490.0,150.0,visa,226.0,debit,220.0,87.0,4.0,null,live.com,null,1.0,1.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,0.0,2.0,0.0,3.0,1.0,19.0,19.0,7.0,0.0,null,null,null,null,null,19.0,19.0,null,null,null,0.0,T,T,F,null,null,F,F,T,T,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,107.9000015258789,0.0,0.0,107.9000015258789,0.0,0.0,48.95000076293945,107.9000015258789,48.95000076293945,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
3356198,0,9172861,107.95,W,16270,404.0,150.0,mastercard,102.0,credit,476.0,87.0,2.0,null,yahoo.com,null,4.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,19.0,4.0,578.0,578.0,18.0,18.0,18.0,null,null,null,null,578.0,528.0,null,null,null,304.0,T,T,T,null,null,T,F,T,T,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.

In [0]:
# Print the schema of the transaction table
df.printSchema()

root
 |-- TransactionID: integer (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- TransactionDT: integer (nullable = true)
 |-- TransactionAmt: double (nullable = true)
 |-- ProductCD: string (nullable = true)
 |-- card1: integer (nullable = true)
 |-- card2: double (nullable = true)
 |-- card3: double (nullable = true)
 |-- card4: string (nullable = true)
 |-- card5: double (nullable = true)
 |-- card6: string (nullable = true)
 |-- addr1: double (nullable = true)
 |-- addr2: double (nullable = true)
 |-- dist1: double (nullable = true)
 |-- dist2: double (nullable = true)
 |-- P_emaildomain: string (nullable = true)
 |-- R_emaildomain: string (nullable = true)
 |-- C1: double (nullable = true)
 |-- C2: double (nullable = true)
 |-- C3: double (nullable = true)
 |-- C4: double (nullable = true)
 |-- C5: double (nullable = true)
 |-- C6: double (nullable = true)
 |-- C7: double (nullable = true)
 |-- C8: double (nullable = true)
 |-- C9: double (nullable = true)
 |-- C10:

In [0]:
%sql
-- Overview of identity table 
SELECT * FROM fraud.bronze.identity_raw LIMIT 5;

TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
2987004,0.0,70787.0,null,null,null,null,null,null,null,null,100.0,NotFound,null,-480.0,New,NotFound,166.0,null,542.0,144.0,null,null,null,null,null,null,null,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
2987008,-5.0,98945.0,null,null,0.0,-5.0,null,null,null,null,100.0,NotFound,49.0,-300.0,New,NotFound,166.0,null,621.0,500.0,null,null,null,null,null,null,null,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,100.0,NotFound,52.0,null,Found,Found,121.0,null,410.0,142.0,null,null,null,null,null,null,null,Found,Found,null,chrome 62.0,null,null,null,F,F,T,T,desktop,Windows
2987011,-5.0,221832.0,null,null,0.0,-6.0,null,null,null,null,100.0,NotFound,52.0,null,New,NotFound,225.0,null,176.0,507.0,null,null,null,null,null,null,null,New,NotFound,null,chrome 62.0,null,null,null,F,F,T,T,desktop,null
2987016,0.0,7460.0,0.0,0.0,1.0,0.0,null,null,0.0,0.0,100.0,NotFound,null,-300.0,Found,Found,166.0,15.0,529.0,575.0,null,null,null,null,null,null,null,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [0]:
#Schema of the identity table
df_identity.printSchema()

root
 |-- TransactionID: integer (nullable = true)
 |-- id_01: double (nullable = true)
 |-- id_02: double (nullable = true)
 |-- id_03: double (nullable = true)
 |-- id_04: double (nullable = true)
 |-- id_05: double (nullable = true)
 |-- id_06: double (nullable = true)
 |-- id_07: double (nullable = true)
 |-- id_08: double (nullable = true)
 |-- id_09: double (nullable = true)
 |-- id_10: double (nullable = true)
 |-- id_11: double (nullable = true)
 |-- id_12: string (nullable = true)
 |-- id_13: double (nullable = true)
 |-- id_14: double (nullable = true)
 |-- id_15: string (nullable = true)
 |-- id_16: string (nullable = true)
 |-- id_17: double (nullable = true)
 |-- id_18: double (nullable = true)
 |-- id_19: double (nullable = true)
 |-- id_20: double (nullable = true)
 |-- id_21: double (nullable = true)
 |-- id_22: double (nullable = true)
 |-- id_23: string (nullable = true)
 |-- id_24: double (nullable = true)
 |-- id_25: double (nullable = true)
 |-- id_26: double (null